<a href="https://colab.research.google.com/github/aymuos/endgame/blob/main/04_cdnod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CDNOD causal discovery with city regimes

This notebook consumes the outputs of `00_master_feature_creator.ipynb` and runs **CDNOD separately at delivery level and batch level**.

- Regime/domain selector: `city` (`Shanghai`, `Hangzhou`, `Chongqing`)
- Delivery target: `eta_mins`
- Batch target: `eta_mean`
- Candidate variables are restricted to the requested causal families: batch size, spatial dispersion, temporal-slack proxies, merchant concentration, and courier workload state.
- Bootstrap stability is used to retain only stable direct causes and stable directed ancestors of delay.

> CDNOD identifies distribution shifts associated with the city index. City is supplied as the special context variable and is not treated as an ordinary operational feature.

## Interpretation guardrails

 This notebook uses leakage-safe pre-delivery workload state (`courier_eta_ewm` and recent GPS motion) and automatically prefers an explicit workload column if one is present in a future export.

`typecode_cb` is the available merchant/type concentration proxy. At batch level its mean is used. Temporal slack is proxied by receipt-time cyclic variables and dispatch position; no promised-deadline field exists in the master output.

In [2]:
# Run once in a fresh Colab/runtime.
!pip install -q causal-learn pyarrow pandas numpy scipy scikit-learn networkx matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.5/245.5 kB 14.0 MB/s eta 0:00:00


In [3]:
import os
from google.colab import drive

# Attempt to unmount and remove the directory if it's in a bad state
if os.path.exists('/content/drive'):
    try:
        # Unmount if already mounted (e.g., from a previous failed attempt)
        os.system('fusermount -uz /content/drive')
        # Remove the directory if it's not empty, which is causing the error
        os.system('rm -rf /content/drive')
    except Exception:
        # Ignore errors if it's not mounted or cannot be removed
        pass

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [20]:
from pathlib import Path
import inspect
import json
import warnings

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

from causallearn.search.ConstraintBased.CDNOD import cdnod

warnings.filterwarnings('ignore', category=RuntimeWarning)
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Master notebook output directory
DATA_DIR = Path('/content/drive/MyDrive/ml/CORRECTEDv3')
OUTPUT_DIR = DATA_DIR / 'causal_graphs' / 'cdnod'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CITY_ORDER = ['Shanghai', 'Hangzhou', 'Chongqing']
CITY_CODE = {city: i for i, city in enumerate(CITY_ORDER)}

ALPHA = 0.01
N_BOOTSTRAPS = 20
STABILITY_THRESHOLD = 0.45
MAX_ANCESTOR_DEPTH = 1 # Set to 1 for faster search
MAX_ROWS_PER_CITY = 500 # Set to 500 for FastKCI stability
MIN_COMPLETE_ROWS = 100

print('CDNOD signature:', inspect.signature(cdnod))
print('Output directory:', OUTPUT_DIR)
print(f'Stability Threshold set to: {STABILITY_THRESHOLD}')
print(f'Max rows per city set to: {MAX_ROWS_PER_CITY}')
print(f'N_BOOTSTRAPS set to: {N_BOOTSTRAPS}')

CDNOD signature: (data: numpy.ndarray, c_indx: numpy.ndarray, alpha: float = 0.05, indep_test: str = 'fisherz', stable: bool = True, uc_rule: int = 0, uc_priority: int = 2, mvcdnod: bool = False, correction_name: str = 'MV_Crtn_Fisher_Z', background_knowledge: Optional[causallearn.utils.PCUtils.BackgroundKnowledge.BackgroundKnowledge] = None, verbose: bool = False, show_progress: bool = True, **kwargs) -> causallearn.graph.GraphClass.CausalGraph
Output directory: /content/drive/MyDrive/ml/CORRECTEDv3/causal_graphs/cdnod
Stability Threshold set to: 0.45
Max rows per city set to: 500
N_BOOTSTRAPS set to: 20


## Restricted causal feature contract

Aliases make the notebook compatible with both full v4 delivery features and filtered/aggregated master outputs. Only one available source is selected for each named concept.

In [14]:
DELIVERY_CONCEPTS = {
    # Batch size / queue position
    'batch_size': ['batch_size'],
    'dispatch_position': ['batch_rank_capped'], # Removed 'batch_rank_dispatch'

    # Spatial dispersion / route geometry
    'spatial_dispersion': [
        'distance_to_batch_centroid',
        'spatial_congestion_norm',
        'nearest_neighbor_distance', # Added
        'same_aoi_share_in_batch'    # Added
    ],
    'route_distance': ['pickup_destination_distance'],
    'remaining_haul': ['remaining_haul_distance'],

    # Temporal slack / demand-window proxies
    'receipt_hour_sin': ['hour_sin'],
    'receipt_hour_cos': ['hour_cos'],
    'holiday_pressure': ['is_holiday', 'is_weekend'],

    # Merchant concentration proxy
    'merchant_concentration': ['typecode_cb'],

    # Courier workload / operational state (all measured before delivery)
    'courier_workload': [ 'workload_causal', 'courier_local_load', 'courier_eta_ewm'],
    'recent_speed': ['speed_mean_15m', 'speed_mean'],
    'recent_motion': ['distance_travelled_15m', 'distance_travelled'],
}

BATCH_CONCEPTS = {
    'batch_size': ['batch_size'],
    'spatial_dispersion': ['batch_grid_cells_unique', 'spatial_congestion_daily_std', 'spatial_congestion_norm_std'],
    'route_distance': ['pickup_destination_distance_mean'],
    'remaining_haul': ['remaining_haul_distance_mean'],
    'receipt_hour_sin': ['hour_sin'],
    'receipt_hour_cos': ['hour_cos'],
    'holiday_pressure': ['is_holiday', 'is_weekend'],
    'merchant_concentration': ['typecode_cb_mean'],
    'courier_workload': ['active_queue_at_receipt', 'workload_causal', 'courier_local_load', 'courier_eta_ewm'],
    'recent_speed': ['speed_mean_15m_mean', 'speed_mean_mean'],
    'recent_motion': ['distance_travelled_15m_mean', 'distance_travelled_mean'],
}

TARGETS = {'delivery': 'eta_mins', 'batch': 'eta_mean'}

In [15]:
def _first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return None


def load_three_city_level(level, data_dir=DATA_DIR):
    """Load master outputs and guarantee a city column."""
    frames, used_paths = [], []

    if level == 'delivery':
        combined = _first_existing([
            data_dir / 'all_cities_delivery_features_v4.parquet',
            data_dir / 'all_cities_v4_filtered.parquet',
        ])
        if combined is not None:
            df = pd.read_parquet(combined)
            if 'city' in df.columns:
                return df, [combined]

        for city in CITY_ORDER:
            slug = city.lower()
            path = _first_existing([
                data_dir / f'delivery_features_{slug}.parquet',
                data_dir / f'delivery_lvl_{slug}_v4.parquet',
            ])
            if path is None:
                raise FileNotFoundError(f'No delivery master output found for {city}')
            part = pd.read_parquet(path)
            part['city'] = city
            frames.append(part)
            used_paths.append(path)

    elif level == 'batch':
        combined = _first_existing([data_dir / 'batch_aggregated_v4.parquet'])
        if combined is not None:
            df = pd.read_parquet(combined)
            if 'city' in df.columns and df['city'].notna().all():
                return df, [combined]

        for city in CITY_ORDER:
            path = data_dir / f'batch_scm_{city.lower()}.parquet'
            if not path.exists():
                raise FileNotFoundError(f'No batch master output found for {city}')
            part = pd.read_parquet(path)
            part['city'] = city
            frames.append(part)
            used_paths.append(path)
    else:
        raise ValueError("level must be 'delivery' or 'batch'")

    return pd.concat(frames, ignore_index=True, sort=False), used_paths


for level in ('delivery',):
    preview, paths = load_three_city_level(level)
    print(f'{level:8s}: {preview.shape} | cities={preview.city.value_counts().to_dict()}')
    print('  sources:', [str(p) for p in paths])

delivery: (100163, 67) | cities={'Hangzhou': 40348, 'Shanghai': 34480, 'Chongqing': 25335}
  sources: ['/content/drive/MyDrive/ml/CORRECTEDv3/delivery_features_shanghai.parquet', '/content/drive/MyDrive/ml/CORRECTEDv3/delivery_features_hangzhou.parquet', '/content/drive/MyDrive/ml/CORRECTEDv3/delivery_features_chongqing.parquet']


In [16]:
def prepare_cdnod_data(level):
    """
    Loads, preprocesses, scales, and samples data for CDNOD analysis.

    Args:
        level (str): 'delivery' or 'batch' to specify which data and concepts to use.

    Returns:
        tuple: (model_df, X, c_indx, variable_names, selected_features_list, paths)
            model_df (pd.DataFrame): The preprocessed DataFrame.
            X (np.ndarray): The feature matrix (including target), scaled.
            c_indx (np.ndarray): The city context indices.
            variable_names (list): Names of features in X.
            selected_features_list (list): List of selected feature names (pre-scaling).
            paths (list): List of file paths from which data was loaded.
    """
    df, paths = load_three_city_level(level)

    concepts_map = DELIVERY_CONCEPTS if level == 'delivery' else BATCH_CONCEPTS
    target = TARGETS[level]

    # Map concepts to actual available features in the DataFrame
    selected_features_list = []
    for concept_name, feature_options in concepts_map.items():
        found = False
        for option in feature_options:
            if option in df.columns:
                selected_features_list.append(option)
                found = True
                break
        if not found and concept_name != 'merchant_concentration':
            # This is a soft warning, CDNOD can run with missing features,
            # but it's good to note if a concept couldn't be represented.
            print(f"Warning: No feature found for concept '{concept_name}' in level '{level}'. Skipping.")

    # Ensure the target variable is in the selected features
    if target not in selected_features_list:
        if target in df.columns:
            selected_features_list.append(target)
        else:
            raise ValueError(f"Target variable '{target}' not found in DataFrame for level '{level}'.")

    # The 'city' column is crucial for c_indx and should always be present
    if 'city' not in df.columns:
        raise ValueError("DataFrame must contain a 'city' column for CDNOD analysis.")

    # Filter DataFrame to only include selected features and 'city', then drop rows with any NaN
    # This ensures a complete dataset for the CDNOD algorithm.
    model_df = df[selected_features_list + ['city']].dropna().copy()

    # Stratified sampling by city to control the total number of rows per city,
    # preventing one city from dominating the dataset if MAX_ROWS_PER_CITY is set.
    if MAX_ROWS_PER_CITY > 0:
        sampled_dfs = []
        for city in CITY_ORDER:
            city_df = model_df[model_df.city == city]
            if len(city_df) > MAX_ROWS_PER_CITY:
                sampled_dfs.append(city_df.sample(n=MAX_ROWS_PER_CITY, random_state=RANDOM_SEED))
            else:
                sampled_dfs.append(city_df)
        model_df = pd.concat(sampled_dfs, ignore_index=True)

    # Check if enough complete rows remain after sampling.
    if len(model_df) < MIN_COMPLETE_ROWS:
        raise ValueError(f"Not enough complete rows after sampling for level {level}. "
                         f"Found {len(model_df)} but need at least {MIN_COMPLETE_ROWS}.")

    # Scale numerical features for CDNOD. 'city' is categorical and will be mapped to c_indx.
    # The target variable is treated as another feature in X for CDNOD.
    numerical_features_to_scale = [f for f in selected_features_list]

    scaler = StandardScaler()
    model_df[numerical_features_to_scale] = scaler.fit_transform(model_df[numerical_features_to_scale])

    # X for CDNOD should contain all features (including the target) to be analyzed
    X_data = model_df[selected_features_list].values
    # c_indx maps city names to numerical codes, which CDNOD uses as context.
    c_indx = model_df.city.map(CITY_CODE).values

    # variable_names are simply the names of the columns in X_data
    variable_names = selected_features_list

    print(f"--- Data Preparation Summary for '{level}' level ---")
    print(f"  Original DataFrame shape: {df.shape}")
    print(f"  Processed DataFrame shape: {model_df.shape}")
    print(f"  Selected features ({len(variable_names)}): {variable_names}")
    print(f"  City counts in processed data: {model_df.city.value_counts().to_dict()}")
    print(f"  Target variable: '{target}'")
    print(f"  Using data from: {[str(p) for p in paths]}")

    return model_df, X_data, c_indx, variable_names, selected_features_list, paths

# Re-executing preparation with updated MAX_ROWS_PER_CITY
model_df, X, c_indx, variable_names, selected, paths = prepare_cdnod_data('delivery')

--- Data Preparation Summary for 'delivery' level ---
  Original DataFrame shape: (100163, 67)
  Processed DataFrame shape: (1500, 13)
  Selected features (12): ['batch_size', 'distance_to_batch_centroid', 'pickup_destination_distance', 'remaining_haul_distance', 'hour_sin', 'hour_cos', 'is_holiday', 'typecode_cb', 'courier_eta_ewm', 'speed_mean_15m', 'distance_travelled_15m', 'eta_mins']
  City counts in processed data: {'Shanghai': 500, 'Hangzhou': 500, 'Chongqing': 500}
  Target variable: 'eta_mins'
  Using data from: ['/content/drive/MyDrive/ml/CORRECTEDv3/delivery_features_shanghai.parquet', '/content/drive/MyDrive/ml/CORRECTEDv3/delivery_features_hangzhou.parquet', '/content/drive/MyDrive/ml/CORRECTEDv3/delivery_features_chongqing.parquet']


In [25]:
print("--- Numeric Type Check ---")
# Check the dtypes of the features selected for the model
print(model_df[selected].dtypes)

# Identify any non-numeric columns in the feature set
non_numeric = model_df[selected].select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print(f"\nWarning: Found non-numeric columns that might cause issues: {non_numeric}")
else:
    print("\nSuccess: All selected features are numeric.")

# Confirm stratification
print("\n--- Stratification Check ---")
print(model_df['city'].value_counts())

--- Numeric Type Check ---
batch_size                     float64
distance_to_batch_centroid     float64
pickup_destination_distance    float64
remaining_haul_distance        float64
hour_sin                       float64
hour_cos                       float64
is_holiday                     float64
typecode_cb                    float64
courier_eta_ewm                float64
speed_mean_15m                 float64
distance_travelled_15m         float64
eta_mins                       float64
dtype: object

Success: All selected features are numeric.

--- Stratification Check ---
city
Shanghai     500
Hangzhou     500
Chongqing    500
Name: count, dtype: int64


## CDNOD runner and graph extraction

The wrapper filters keyword arguments against the installed `causal-learn` signature. Graph endpoint conventions follow causal-learn: `-1` is a tail, `1` an arrowhead, and `2` a circle.

In [29]:
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode

def run_cdnod(X, c_indx, alpha=ALPHA, show_progress=False, variable_names=None):
    # Numerical Stability: Add tiny jitter to prevent FastKCI 'zero-size array' errors on near-duplicates
    X_stable = X + np.random.normal(0, 1e-9, X.shape)

    # CDNOD nodes: [features] + [target] + [city_context]
    num_vars = X_stable.shape[1] + 1
    nodes = [GraphNode(f'node{i}') for i in range(num_vars)]

    bk = BackgroundKnowledge()
    if variable_names is not None:
        target_idx = len(variable_names) - 1
        city_idx = len(variable_names)

        # 1. Forbid Target -> Features
        for i in range(target_idx):
            bk.add_forbidden_by_node(nodes[target_idx], nodes[i])

        # 2. Forbid Features -> City Context
        for i in range(city_idx):
            bk.add_forbidden_by_node(nodes[i], nodes[city_idx])

        # 3. Temporal Tiers
        tier_1 = {'is_holiday', 'is_weekend', 'hour_sin', 'hour_cos'}
        for i, name in enumerate(variable_names[:-1]):
            if name in tier_1:
                for j, other_name in enumerate(variable_names[:-1]):
                    if other_name not in tier_1:
                        bk.add_forbidden_by_node(nodes[j], nodes[i])

    requested = {
        'alpha': alpha,
        'indep_test': 'fastkci',
        'stable': True,
        'uc_rule': 0,
        'depth': MAX_ANCESTOR_DEPTH,
        'uc_priority': 2,
        'show_progress': show_progress,
        'background_knowledge': bk
    }
    supported = inspect.signature(cdnod).parameters
    kwargs = {k: v for k, v in requested.items() if k in supported}

    return cdnod(X_stable, c_indx.reshape(-1, 1), **kwargs)

def graph_matrix(cg):
    graph_obj = getattr(cg, 'G', cg)
    matrix = getattr(graph_obj, 'graph', None)
    if matrix is None:
        raise AttributeError('Could not locate causal-learn graph matrix')
    return np.asarray(matrix)

def endpoint_edge(matrix, i, j):
    a, b = int(matrix[i, j]), int(matrix[j, i])
    if a == 0 and b == 0: return None
    if a == -1 and b == 1: return 'i_to_j'
    if a == 1 and b == -1: return 'j_to_i'
    return 'ambiguous'

def extract_edges(cg, variable_names):
    matrix = graph_matrix(cg)
    names = list(variable_names)
    if matrix.shape[0] == len(names) + 1: names.append('city_context')
    rows = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            kind = endpoint_edge(matrix, i, j)
            if kind == 'i_to_j':
                rows.append((names[i], names[j], 'directed'))
            elif kind == 'j_to_i':
                rows.append((names[j], names[i], 'directed'))
            elif kind == 'ambiguous':
                rows.append((names[i], names[j], 'ambiguous'))
    return pd.DataFrame(rows, columns=['source', 'target', 'edge_type']), names

### Independence Test Configuration
The `indep_test` parameter in CDNOD determines how conditional independence is calculated.
- `fastkci`: Slower, non-parametric (nonlinear).

In [27]:
def stratified_bootstrap_indices(city_labels, rng):
    idx = []
    city_labels = np.asarray(city_labels)
    for city in CITY_ORDER:
        city_idx = np.flatnonzero(city_labels == city)
        idx.extend(rng.choice(city_idx, size=len(city_idx), replace=True))
    return np.asarray(idx, dtype=int)

def bootstrap_cdnod(X, c_indx, city_labels, variable_names, n_bootstraps=N_BOOTSTRAPS):
    records = []
    successful_bootstraps = 0
    local_rng = np.random.default_rng(RANDOM_SEED)

    for b in range(n_bootstraps):
        idx = stratified_bootstrap_indices(city_labels, local_rng)
        try:
            # Pass variable_names to apply background knowledge constraints
            cg_b = run_cdnod(X[idx], c_indx[idx], show_progress=False, variable_names=variable_names)
            edges_b, _ = extract_edges(cg_b, variable_names)
            successful_bootstraps += 1
            for row in edges_b.itertuples(index=False):
                pair = tuple(sorted((row.source, row.target)))
                records.append({
                    'bootstrap': b,
                    'node_a': pair[0],
                    'node_b': pair[1],
                    'source': row.source,
                    'target': row.target,
                    'edge_type': row.edge_type,
                })
        except Exception as exc:
            print(f'Bootstrap {b} failed: {exc}')

        if (b + 1) % 10 == 0 or b == n_bootstraps - 1:
            print(f'  completed {b + 1}/{n_bootstraps} bootstraps')

    rec = pd.DataFrame(records)
    if rec.empty: return pd.DataFrame()
    completed = max(successful_bootstraps, 1)
    adjacency = rec.groupby(['node_a', 'node_b']).bootstrap.nunique().div(completed).rename('adjacency_stability').reset_index()
    directed = rec[rec.edge_type == 'directed']
    if directed.empty:
        adjacency['best_source'], adjacency['best_target'], adjacency['direction_stability'] = None, None, 0.0
        return adjacency
    direction_counts = directed.groupby(['node_a', 'node_b', 'source', 'target']).bootstrap.nunique().rename('directed_count').reset_index().sort_values('directed_count', ascending=False).drop_duplicates(['node_a', 'node_b'])
    result = adjacency.merge(direction_counts, on=['node_a', 'node_b'], how='left')
    result['direction_stability'] = result['directed_count'].fillna(0) / completed
    return result.rename(columns={'source': 'best_source', 'target': 'best_target'}).drop(columns=['directed_count'])

In [28]:
def select_delay_ancestors(stability, target, threshold=STABILITY_THRESHOLD, max_depth=MAX_ANCESTOR_DEPTH):
    """Keep stable directed parents/ancestors of delay; report stable ambiguous neighbors separately."""
    stable = stability[stability.adjacency_stability >= threshold].copy()
    directed = stable[
        stable.best_source.notna()
        & (stable.direction_stability >= threshold)
    ].copy()

    graph = nx.DiGraph()
    graph.add_edges_from(directed[['best_source', 'best_target']].itertuples(index=False, name=None))

    kept = {target}
    direct_parents = set(graph.predecessors(target)) if target in graph else set()
    kept.update(direct_parents)

    frontier = set(direct_parents)
    for _ in range(max_depth - 1):
        next_frontier = set()
        for node in frontier:
            next_frontier.update(graph.predecessors(node))
        next_frontier -= kept
        kept.update(next_frontier)
        frontier = next_frontier
        if not frontier:
            break

    ambiguous_neighbors = set()
    for row in stable.itertuples(index=False):
        if target in {row.node_a, row.node_b} and row.direction_stability < threshold:
            ambiguous_neighbors.add(row.node_b if row.node_a == target else row.node_a)

    kept.discard('city_context')
    direct_parents.discard('city_context')
    return sorted(kept), sorted(direct_parents), sorted(ambiguous_neighbors), directed

def plot_full_cpdag(stability, title, threshold=STABILITY_THRESHOLD):
    """Visualizes full learned structure with prominent arrows and confidence labels."""
    stable = stability[stability.adjacency_stability >= threshold].copy()
    G = nx.DiGraph()

    for row in stable.itertuples(index=False):
        if row.best_source and row.direction_stability >= threshold:
            G.add_edge(row.best_source, row.best_target, style='solid', weight=row.direction_stability)
        else:
            G.add_edge(row.node_a, row.node_b, style='dashed', weight=row.adjacency_stability)

    if not G.nodes:
        print("No stable edges found.")
        return

    plt.figure(figsize=(18, 12))
    pos = nx.kamada_kawai_layout(G)

    target_nodes = [n for n in G.nodes if 'eta' in n]
    context_nodes = [n for n in G.nodes if n == 'city_context']
    other_nodes = [n for n in G.nodes if n not in target_nodes and n not in context_nodes]

    nx.draw_networkx_nodes(G, pos, nodelist=target_nodes, node_color='tomato', node_size=4000, label='Target')
    nx.draw_networkx_nodes(G, pos, nodelist=context_nodes, node_color='gold', node_size=4000, label='Regime')
    nx.draw_networkx_nodes(G, pos, nodelist=other_nodes, node_color='skyblue', node_size=3200)

    directed_edges = [e for e, d in G.edges.items() if d['style'] == 'solid']
    undirected_edges = [e for e, d in G.edges.items() if d['style'] == 'dashed']

    # Drawing with larger arrows
    nx.draw_networkx_edges(G, pos, edgelist=directed_edges, width=2.5, arrowsize=35, min_source_margin=20, min_target_margin=20)
    nx.draw_networkx_edges(G, pos, edgelist=undirected_edges, width=1.5, style='dashed', alpha=0.6, arrows=False)

    # Prominent Confidence Labels
    edge_labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=11, font_color='black', font_weight='bold', bbox=dict(alpha=0.7, color='white'))

    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')
    plt.title(f"{title}\n(Labels = Confidence/Stability | Threshold={threshold})", fontsize=16)
    plt.axis('off')
    plt.show()

In [23]:
def run_level_pipeline(level):
    model_df, X, c_indx, variable_names, selected, paths = prepare_cdnod_data(level)
    target = TARGETS[level]

    print(f'\nRunning full-sample CDNOD for {level} level...')
    cg = run_cdnod(X, c_indx, show_progress=True, variable_names=variable_names)
    full_edges, graph_names = extract_edges(cg, variable_names)

    print(f'\nEstimating edge stability with {N_BOOTSTRAPS} stratified bootstraps...')
    stability = bootstrap_cdnod(X, c_indx, model_df.city.to_numpy(), variable_names)
    stability = stability.sort_values(['adjacency_stability', 'direction_stability'], ascending=False)

    kept, direct_parents, ambiguous_neighbors, stable_directed = select_delay_ancestors(stability, target)
    ancestor_features = [c for c in kept if c != target]

    print('\n--- FULL CPDAG VISUALIZATION ---')
    plot_full_cpdag(stability, f'Full CDNOD CPDAG — {level.title()}')

    print('\nStable direct causes/parents of delay:', direct_parents or 'none oriented')
    print('Stable ancestors retained:', ancestor_features or 'none oriented')
    print('Stable but unoriented delay neighbors:', ambiguous_neighbors or 'none')

    # Saving logic remains consistent...
    reduced = model_df[['city'] + kept].copy()
    stability.to_csv(OUTPUT_DIR / f'{level}_cdnod_edge_stability.csv', index=False)

    return {
        'stability': stability,
        'kept': kept,
        'direct_parents': direct_parents,
        'ambiguous_neighbors': ambiguous_neighbors
    }

## 1. Delivery-level CDNOD

Each row is an order. The resulting reduced parquet contains city, delivery delay, and only stable direct parents/ancestors of delivery delay.

In [ ]:
# Execute delivery-level CDNOD with fastkci independence test
delivery_results = run_level_pipeline('delivery')

--- Data Preparation Summary for 'delivery' level ---
  Original DataFrame shape: (100163, 67)
  Processed DataFrame shape: (1500, 13)
  Selected features (12): ['batch_size', 'distance_to_batch_centroid', 'pickup_destination_distance', 'remaining_haul_distance', 'hour_sin', 'hour_cos', 'is_holiday', 'typecode_cb', 'courier_eta_ewm', 'speed_mean_15m', 'distance_travelled_15m', 'eta_mins']
  City counts in processed data: {'Shanghai': 500, 'Hangzhou': 500, 'Chongqing': 500}
  Target variable: 'eta_mins'
  Using data from: ['/content/drive/MyDrive/ml/CORRECTEDv3/delivery_features_shanghai.parquet', '/content/drive/MyDrive/ml/CORRECTEDv3/delivery_features_hangzhou.parquet', '/content/drive/MyDrive/ml/CORRECTEDv3/delivery_features_chongqing.parquet']

Running full-sample CDNOD for delivery level...


  0%|          | 0/13 [00:00<?, ?it/s]


Estimating edge stability with 20 stratified bootstraps...


  0%|          | 0/13 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]

...

  0%|          | 0/13 [00:00<?, ?it/s]

Bootstrap 12 failed: zero-size array to reduction operation maximum which has no identity


  0%|          | 0/13 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]

In [ ]:
# Final summary of delivery results
if 'delivery_results' in globals():
    print("--- Top 15 Stable Causal Edges (Delivery) ---")
    display(delivery_results['stability'].head(15))

    print(f"\nRetained {len(delivery_results['kept'])} features as stable delay ancestors.")
    print(f"Direct Parents: {delivery_results['direct_parents']}")
    print(f"Ambiguous Neighbors: {delivery_results['ambiguous_neighbors']}")

In [ ]:
def plot_stable_ancestors(directed, kept, target, title):
    sub = directed[
        directed.best_source.isin(kept + ['city_context'])
        & directed.best_target.isin(kept + ['city_context'])
    ]
    G = nx.DiGraph()
    for row in sub.itertuples(index=False):
        G.add_edge(row.best_source, row.best_target, weight=row.direction_stability)

    if not G.nodes:
        print('No stable directed delay-ancestor graph at this threshold.')
        return

    plt.figure(figsize=(11, 7))
    pos = nx.spring_layout(G, seed=RANDOM_SEED, k=1.2)
    colors = ['tomato' if n == target else ('gold' if n == 'city_context' else 'skyblue') for n in G.nodes]
    widths = [1 + 4 * G[u][v]['weight'] for u, v in G.edges]

    nx.draw_networkx(G, pos, node_color=colors, node_size=2300, width=widths, arrowsize=22, font_size=9, font_weight='bold')
    labels = {(u, v): f"{G[u][v]['weight']:.2f}" for u, v in G.edges}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_size=8)
    plt.title(title, fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Visualize specifically for the delivery results
if 'delivery_results' in globals():
    print('\n--- STABLE DELAY ANCESTORS ---')
    plot_stable_ancestors(
        delivery_results['stability'][delivery_results['stability'].direction_stability >= STABILITY_THRESHOLD],
        delivery_results['kept'],
        TARGETS['delivery'],
        'CDNOD Stable Delay Ancestors — Delivery Level'
    )

In [ ]:
if 'delivery_results' in globals():
    plot_full_cpdag(
        delivery_results['stability'],
        'Full CDNOD CPDAG — Delivery Level',
        threshold=STABILITY_THRESHOLD
    )

In [ ]:
print('--- Top Edge Stability Scores ---')
if 'delivery_results' in globals():
    display(delivery_results['stability'].sort_values('adjacency_stability', ascending=False).head(10))
else:
    print('delivery_results not found. Please ensure the delivery-run cell finished executing.')

## 2. Batch-level CDNOD

Each row is a courier batch. The target is mean batch delay (`eta_mean`), and dispersion/workload variables use batch-level aggregates from the master notebook.

In [ ]:
# batch_results = run_level_pipeline('batch')

## Cross-level comparison

Features retained at both levels are the most portable causal candidates. Level-specific ancestors may reflect within-order routing effects versus batch composition effects.

In [ ]:
delivery_ancestors = set(delivery_results['kept']) - {TARGETS['delivery']}
batch_ancestors = set(batch_results['kept']) - {TARGETS['batch']}

comparison = pd.DataFrame({
    'feature': sorted(delivery_ancestors | batch_ancestors),
}).assign(
    delivery_ancestor=lambda d: d.feature.isin(delivery_ancestors),
    batch_ancestor=lambda d: d.feature.isin(batch_ancestors),
    retained_at_both=lambda d: d.delivery_ancestor & d.batch_ancestor,
)

comparison_path = OUTPUT_DIR / 'cdnod_delivery_batch_ancestor_comparison.csv'
comparison.to_csv(comparison_path, index=False)
display(comparison)
print('Saved:', comparison_path)

## Reading the outputs

- `*_cdnod_delay_ancestors.parquet`: final reduced modeling table.
- `*_cdnod_edge_stability.csv`: bootstrap adjacency and direction frequencies.
- `*_cdnod_metadata.json`: selected source columns, thresholds, direct parents, ancestors, and ambiguous delay neighbors.
- A stable `city_context → feature` edge means that mechanism/distribution changes across cities; it is not itself an operational intervention.
- Stable but unoriented neighbors are reported, but deliberately excluded from the strict ancestor table. Lower `STABILITY_THRESHOLD` only as a sensitivity analysis.